## Side-by-Side Benchmark: EasyOCR vs PaddleOCR vs Groq Vision VLM

This section runs the **same pages** from `sample_data/student_sheet_1.pdf` through all three engines and compares:

- extraction latency, character/word volume, and detected text segments
- mean OCR confidence (available for EasyOCR and PaddleOCR)
- keyword coverage against the labelled keywords in `sample_data/questions.csv` (fuzzy match, 85% threshold)
- cross-engine text agreement (`difflib.SequenceMatcher`)
- a side-by-side preview of the extracted text for each page

| Engine | Input handling | Notes |
|---|---|---|
| **EasyOCR** | Same raster pages, `canvas_size=3200`, GPU if available | Traditional detector + recognizer with bounding boxes |
| **PaddleOCR** | Same raster pages, PP-OCRv6, CPU | oneDNN disabled and thread count capped for stability |
| **Groq VLM** | Same raster pages resized to max 2048 px, JPEG q=90 | `qwen/qwen3.8-27b` via Groq API |

The section is self-contained and can be executed independently of the exploratory cells above.

In [12]:
# =============================================================================
# Benchmark configuration and shared dataset
# =============================================================================
import base64
import io
import os
import textwrap
import time
from pathlib import Path

import pandas as pd
import pymupdf
from PIL import Image
from rapidfuzz import fuzz

PDF_PATH = Path("sample_data/student_sheet_1.pdf")
QA_PATH = Path("sample_data/questions.csv")

BENCHMARK_DPI = 150
EASYOCR_CANVAS_SIZE = 3200
GROQ_MODEL = "qwen/qwen3.8-27b"
GROQ_MAX_DIM = 2048
KEYWORD_MATCH_THRESHOLD = 85.0

# Load every page once so all engines receive identical raster inputs.
doc = pymupdf.open(PDF_PATH)
benchmark_pages = []
for page in doc:
    pix = page.get_pixmap(dpi=BENCHMARK_DPI)
    benchmark_pages.append(
        Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    )

# Ground-truth keywords and marks from the labelled question bank.
qa_df = pd.read_csv(QA_PATH)
reference_keys = {
    str(row.question_id): {
        "keywords": [
            keyword.strip()
            for keyword in str(row.key_words).split(",")
            if keyword.strip()
        ],
        "max_marks": float(row.max_marks),
    }
    for row in qa_df.itertuples()
}

benchmark_results = []
extracted_texts = {}

print(f"Loaded {len(benchmark_pages)} page(s) at {BENCHMARK_DPI} DPI")
print(f"Page dimensions: {[image.size for image in benchmark_pages]}")
print(f"Reference questions: {list(reference_keys)}")

Loaded 2 page(s) at 150 DPI
Page dimensions: [(5034, 7167), (5034, 7067)]
Reference questions: ['Q1', 'Q2']


In [13]:
# -----------------------------------------------------------------------------
# Shared scoring / record helpers
# -----------------------------------------------------------------------------
def normalize_text(text):
    normalized = " ".join((text or "").lower().split())
    for symbol in ("=", "+", "/", "^"):
        normalized = (
            normalized.replace(f" {symbol} ", symbol)
            .replace(f" {symbol}", symbol)
            .replace(f"{symbol} ", symbol)
        )
    return normalized


def score_keywords(text, keywords, threshold=KEYWORD_MATCH_THRESHOLD):
    normalized = normalize_text(text)
    scores = {
        keyword: float(
            fuzz.partial_ratio(normalize_text(keyword), normalized)
        )
        for keyword in keywords
    }
    matched = [keyword for keyword, score in scores.items() if score >= threshold]
    coverage = len(matched) / len(keywords) if keywords else 0.0
    return coverage, matched, {key: round(value, 1) for key, value in scores.items()}


def make_record(
    engine,
    page_number,
    text,
    latency_s,
    confidences=None,
    segments=None,
    error=None,
):
    text = text or ""
    question_id = f"Q{page_number}"
    keywords = reference_keys.get(question_id, {}).get("keywords", [])
    coverage, matched, keyword_scores = score_keywords(text, keywords)
    confidences = confidences or []
    mean_confidence = (
        float(sum(confidences) / len(confidences)) if confidences else None
    )

    return {
        "engine": engine,
        "page": page_number,
        "question_id": question_id,
        "status": "ok" if error is None else "error",
        "latency_s": float(latency_s),
        "chars": len(text),
        "words": len(text.split()),
        "segments": segments,
        "mean_confidence": mean_confidence,
        "keyword_coverage": coverage,
        "matched_keywords": matched,
        "keyword_scores": keyword_scores,
        "text": text,
        "error": error,
    }


def report_record(record):
    confidence = (
        f"{record['mean_confidence'] * 100:.1f}%"
        if record["mean_confidence"] is not None
        else "n/a"
    )
    print(
        f"Page {record['page']}: {record['status']} | "
        f"{record['latency_s']:.2f}s | "
        f"{record['words']} words | "
        f"confidence {confidence} | "
        f"keyword coverage {record['keyword_coverage'] * 100:.0f}%"
    )
    if record["error"]:
        print(f"  ERROR: {record['error']}")

In [14]:
# =============================================================================
# Engine 1: EasyOCR (traditional detector + recognizer)
# =============================================================================
import numpy as np
import easyocr
import torch

EASYOCR_ENGINE = "EasyOCR"
easyocr_reader = None
start = time.perf_counter()
try:
    easyocr_reader = easyocr.Reader(["en"], gpu=True)
    print(f"EasyOCR initialized in {time.perf_counter() - start:.2f}s")
except Exception as exc:
    print(f"EasyOCR initialization failed: {type(exc).__name__}: {exc}")

for page_number, image in enumerate(benchmark_pages, start=1):
    start = time.perf_counter()
    if easyocr_reader is None:
        record = make_record(
            EASYOCR_ENGINE,
            page_number,
            "",
            0.0,
            error="EasyOCR reader could not be initialized.",
        )
    else:
        try:
            result = easyocr_reader.readtext(
                np.array(image),
                detail=1,
                canvas_size=EASYOCR_CANVAS_SIZE,
                mag_ratio=1.0,
                paragraph=False,
            )
            latency = time.perf_counter() - start
            texts = [item[1] for item in result]
            confidences = [float(item[2]) for item in result]
            text = "\n".join(texts)
            record = make_record(
                EASYOCR_ENGINE,
                page_number,
                text,
                latency,
                confidences=confidences,
                segments=len(result),
            )
        except Exception as exc:
            record = make_record(
                EASYOCR_ENGINE,
                page_number,
                "",
                time.perf_counter() - start,
                error=f"{type(exc).__name__}: {exc}",
            )

    benchmark_results.append(record)
    extracted_texts[(EASYOCR_ENGINE, page_number)] = record["text"]
    report_record(record)

    # Release cached CUDA blocks before rasterising the next full page.
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

EasyOCR initialized in 3.95s


Page 1: ok | 2.53s | 136 words | confidence 43.3% | keyword coverage 40%


Page 2: ok | 1.92s | 97 words | confidence 50.0% | keyword coverage 0%


In [15]:
# =============================================================================
# Engine 2: PaddleOCR (PP-OCRv6)
# =============================================================================
import os

os.environ["FLAGS_use_mkldnn"] = "0"
os.environ["PADDLEX_ENABLE_MKLDNN"] = "0"
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

import numpy as np
from paddleocr import PaddleOCR

PADDLE_ENGINE = "PaddleOCR"
paddle_engine = None
start = time.perf_counter()
try:
    paddle_engine = PaddleOCR(
        lang="en",
        device="cpu",
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )
    print(f"PaddleOCR initialized in {time.perf_counter() - start:.2f}s")
except Exception as exc:
    print(f"PaddleOCR initialization failed: {type(exc).__name__}: {exc}")

for page_number, image in enumerate(benchmark_pages, start=1):
    start = time.perf_counter()
    if paddle_engine is None:
        record = make_record(
            PADDLE_ENGINE,
            page_number,
            "",
            0.0,
            error="PaddleOCR engine could not be initialized.",
        )
    else:
        try:
            result = paddle_engine.predict(np.array(image))
            latency = time.perf_counter() - start
            page_result = result[0]
            texts = [str(text) for text in page_result.get("rec_texts", [])]
            confidences = [
                float(score) for score in page_result.get("rec_scores", [])
            ]
            text = "\n".join(texts)
            record = make_record(
                PADDLE_ENGINE,
                page_number,
                text,
                latency,
                confidences=confidences,
                segments=len(texts),
            )
        except Exception as exc:
            record = make_record(
                PADDLE_ENGINE,
                page_number,
                "",
                time.perf_counter() - start,
                error=f"{type(exc).__name__}: {exc}",
            )

    benchmark_results.append(record)
    extracted_texts[(PADDLE_ENGINE, page_number)] = record["text"]
    report_record(record)

Creating model: ('PP-OCRv6_medium_det', None, None)


Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/nithan-b-s/.paddlex/official_models/PP-OCRv6_medium_det`.


/home/nithan-b-s/Documents/collegepur/task_ocr/.venv/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
PLEASE USE OMP_NUM_THREADS WISELY.


Creating model: ('PP-OCRv6_medium_rec', None, None)


Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/nithan-b-s/.paddlex/official_models/PP-OCRv6_medium_rec`.


Resized image size (5034x7167) exceeds max_side_limit of 4000. Resizing to fit within limit.


... (10 outputs left out)

ReduceMeanCheckIfOneDNNSupport


Resized image size (5034x7067) exceeds max_side_limit of 4000. Resizing to fit within limit.


Page 1: ok | 38.32s | 50 words | confidence 94.7% | keyword coverage 40%


Page 2: ok | 38.20s | 56 words | confidence 91.7% | keyword coverage 75%


In [16]:
# =============================================================================
# Engine 3: Groq Vision VLM
# =============================================================================
import dotenv
from groq import Groq

GROQ_ENGINE = "Groq VLM (qwen/qwen3.8-27b)"
GROQ_PROMPT = (
    "Extract all handwritten and printed text from this answer sheet page verbatim. "
    "Include question numbers, formulas, and equations. "
    "Return only the extracted text without explanations."
)

dotenv.load_dotenv()
groq_client = None
start = time.perf_counter()
try:
    groq_client = Groq()
    print(f"Groq client initialized in {time.perf_counter() - start:.2f}s")
except Exception as exc:
    print(f"Groq client initialization failed: {type(exc).__name__}: {exc}")

for page_number, image in enumerate(benchmark_pages, start=1):
    start = time.perf_counter()
    if groq_client is None:
        record = make_record(
            GROQ_ENGINE,
            page_number,
            "",
            0.0,
            error="Groq client could not be initialized.",
        )
    else:
        try:
            page_image = image.copy()
            if max(page_image.size) > GROQ_MAX_DIM:
                page_image.thumbnail(
                    (GROQ_MAX_DIM, GROQ_MAX_DIM),
                    Image.Resampling.LANCZOS,
                )

            buffer = io.BytesIO()
            page_image.save(buffer, format="JPEG", quality=90)
            base64_image = base64.b64encode(buffer.getvalue()).decode("utf-8")

            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": GROQ_PROMPT},
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:image/jpeg;base64,{base64_image}"
                                },
                            },
                        ],
                    }
                ],
            )
            latency = time.perf_counter() - start
            text = response.choices[0].message.content or ""
            record = make_record(
                GROQ_ENGINE,
                page_number,
                text,
                latency,
            )
        except Exception as exc:
            record = make_record(
                GROQ_ENGINE,
                page_number,
                "",
                time.perf_counter() - start,
                error=f"{type(exc).__name__}: {exc}",
            )

    benchmark_results.append(record)
    extracted_texts[(GROQ_ENGINE, page_number)] = record["text"]
    report_record(record)

Groq client initialized in 0.04s


Page 1: ok | 1.58s | 123 words | confidence n/a | keyword coverage 80%


Page 2: ok | 1.22s | 86 words | confidence n/a | keyword coverage 100%


In [17]:
# =============================================================================
# Side-by-side comparison
# =============================================================================
from difflib import SequenceMatcher
from itertools import combinations

pd.set_option("display.max_colwidth", 100)

results_df = pd.DataFrame(benchmark_results)
results_df["keyword_coverage_pct"] = (results_df["keyword_coverage"] * 100).round(1)
results_df["mean_confidence_pct"] = results_df["mean_confidence"] * 100

per_page_df = results_df[
    [
        "engine",
        "page",
        "question_id",
        "status",
        "latency_s",
        "chars",
        "words",
        "segments",
        "mean_confidence_pct",
        "keyword_coverage_pct",
    ]
].copy()
per_page_df["latency_s"] = per_page_df["latency_s"].round(2)
per_page_df["mean_confidence_pct"] = per_page_df["mean_confidence_pct"].round(1)

print("PER-PAGE RESULTS")
display(per_page_df)

aggregate_df = (
    results_df.groupby("engine", sort=False)
    .agg(
        pages=("page", "count"),
        total_latency_s=("latency_s", "sum"),
        avg_latency_s=("latency_s", "mean"),
        total_words=("words", "sum"),
        mean_keyword_coverage_pct=("keyword_coverage_pct", "mean"),
        mean_confidence_pct=(
            "mean_confidence_pct",
            lambda values: round(values.dropna().mean(), 1)
            if values.notna().any()
            else None,
        ),
        errors=("status", lambda values: int((values != "ok").sum())),
    )
    .reset_index()
)
aggregate_df["total_latency_s"] = aggregate_df["total_latency_s"].round(2)
aggregate_df["avg_latency_s"] = aggregate_df["avg_latency_s"].round(2)
aggregate_df["mean_keyword_coverage_pct"] = aggregate_df[
    "mean_keyword_coverage_pct"
].round(1)
aggregate_df = aggregate_df.sort_values(
    ["mean_keyword_coverage_pct", "total_latency_s"],
    ascending=[False, True],
)

print("AGGREGATE RANKING")
display(aggregate_df)

keyword_rows = []
for record in benchmark_results:
    for keyword, score in record["keyword_scores"].items():
        keyword_rows.append(
            {
                "page": record["page"],
                "keyword": keyword,
                "engine": record["engine"],
                "score": score,
            }
        )

if keyword_rows:
    keyword_df = pd.DataFrame(keyword_rows)
    keyword_matrix = keyword_df.pivot_table(
        index=["page", "keyword"],
        columns="engine",
        values="score",
    ).round(1)
    print("KEYWORD SIMILARITY MATRIX")
    display(keyword_matrix)

engine_names = list(results_df["engine"].unique())
agreement_rows = []
for page_number in sorted(results_df["page"].unique()):
    page_texts = {
        engine: results_df.loc[
            (results_df["engine"] == engine) & (results_df["page"] == page_number),
            "text",
        ].iloc[0]
        for engine in engine_names
    }
    for engine_a, engine_b in combinations(engine_names, 2):
        similarity = SequenceMatcher(
            None,
            normalize_text(page_texts[engine_a])[:4000],
            normalize_text(page_texts[engine_b])[:4000],
        ).ratio()
        agreement_rows.append(
            {
                "page": page_number,
                "engine_a": engine_a,
                "engine_b": engine_b,
                "text_similarity_pct": round(similarity * 100, 1),
            }
        )

if agreement_rows:
    print("CROSS-ENGINE TEXT AGREEMENT")
    display(pd.DataFrame(agreement_rows))

PREVIEW_CHARS = 1200
for page_number in sorted(results_df["page"].unique()):
    print("\n" + "=" * 100)
    print(f"PAGE {page_number} | reference keywords: {reference_keys.get(f'Q{page_number}', {}).get('keywords', [])}")
    print("=" * 100)
    for engine in engine_names:
        record = results_df.loc[
            (results_df["engine"] == engine) & (results_df["page"] == page_number)
        ].iloc[0]
        print(
            f"\n--- {engine} | status={record['status']} | "
            f"latency={record['latency_s']:.2f}s | "
            f"keyword coverage={record['keyword_coverage'] * 100:.0f}% ---"
        )
        preview = record["text"][:PREVIEW_CHARS]
        if not preview and record["error"]:
            preview = f"[ERROR] {record['error']}"
        print(textwrap.indent(preview or "[no text extracted]", "  "))

print("\nBenchmark complete.")

PER-PAGE RESULTS


,engine,page,question_id,status,latency_s,chars,words,segments,mean_confidence_pct,keyword_coverage_pct
0,EasyOCR,1,Q1,ok,2.53,704,136,107.0,43.3,40.0
1,EasyOCR,2,Q2,ok,1.92,461,97,73.0,50.0,0.0
2,PaddleOCR,1,Q1,ok,38.32,347,50,13.0,94.7,40.0
3,PaddleOCR,2,Q2,ok,38.20,312,56,10.0,91.7,75.0
4,Groq VLM (qwen/qwen3.8-27b),1,Q1,ok,1.58,766,123,NaN,NaN,80.0
5,Groq VLM (qwen/qwen3.8-27b),2,Q2,ok,1.22,487,86,NaN,NaN,100.0


AGGREGATE RANKING


,engine,pages,total_latency_s,avg_latency_s,total_words,mean_keyword_coverage_pct,mean_confidence_pct,errors
2,Groq VLM (qwen/qwen3.8-27b),2,2.80,1.40,209,90.0,NaN,0
1,PaddleOCR,2,76.52,38.26,106,57.5,93.2,0
0,EasyOCR,2,4.45,2.23,233,20.0,46.6,0


KEYWORD SIMILARITY MATRIX


engine               EasyOCR  Groq VLM (qwen/qwen3.8-27b)  PaddleOCR
page keyword                                                        
1    carbon dioxide     85.7                        100.0       42.9
     glucose            85.7                        100.0       85.7
     light energy       50.0                         66.7       58.3
     photosynthesis     64.0                        100.0      100.0
     water              80.0                        100.0       60.0
2    F=ma               50.0                        100.0      100.0
     acceleration       66.7                        100.0       91.7
     force              80.0                        100.0      100.0
     mass               75.0                        100.0       75.0

CROSS-ENGINE TEXT AGREEMENT


,page,engine_a,engine_b,text_similarity_pct
0,1,EasyOCR,PaddleOCR,4.6
1,1,EasyOCR,Groq VLM (qwen/qwen3.8-27b),2.1
2,1,PaddleOCR,Groq VLM (qwen/qwen3.8-27b),37.5
3,2,EasyOCR,PaddleOCR,2.1
4,2,EasyOCR,Groq VLM (qwen/qwen3.8-27b),6.4
5,2,PaddleOCR,Groq VLM (qwen/qwen3.8-27b),14.5



PAGE 1 | reference keywords: ['photosynthesis', 'light energy', 'glucose', 'carbon dioxide', 'water']

--- EasyOCR | status=ok | latency=2.53s | keyword coverage=40% ---
  Phohvsyntkenc
  i4
  the
  pocer
  by
  wtucA
  Jveen
  the>
  Own
  foorl (glcose)
  from
  Cambon
  dioxi de_
  (cox)
  and
  Webe (1,e)
  in
  Kne
  pre sCete
  ZuubrLt
  and
  borophyl ,
  TMQvTJ
  Of
  PtbTusyANnI ESIf
  I. Cocbon
  droride Cco,)
  Dbtatned
  Krorr
  Aae
  atmee 
  mreval
  Cb mala
  In
  leevek .
  2 .
  wateCuzD)
  AblDrbed
  Knom
  hp
  ba
  ie
  ~bts
  8
  Sunbcat
  Pxv &ea
  te
  needed
  tpr Phetulyntexs
  4 ,
  cblorvphy I) ,
  Mo
  Jreen
  Pijment
  in
  leexex
  hat
  aboobs
  Sunlght
  OuTPvT
  Df
  PlbnoSYATHERtS
  I.Glucoje
  (Clpz %c):
  The
  pyduced
  VJ Ie
  (ox)
  Releeled
  Inh
  kRe
  atusphee
  6 00p
  $ 6
  Snbcy -t
  49
  Choto
  rplj "
  Phwiyynthes ' f
  13
  Cxexkcl
  becalue
  iF
  provca&
  feod
  Ha-t
  Gnd
  releesu
  OX
  yJe
  heeded
  '8
  0~ janjrs
  pr epore
  